# QC and filter seurat object

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        
        # Plotting 
        library(ggplot2), 
        library(patchwork), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.10.16-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Source files
source("plotting_global.R")
source("bin/seurat_qc.R")

# Parameter settings

In [ ]:
# Files 
raw_rds_file <- "data/object/raw.rds"
qc_rds_file <- "data/object/qc.rds"
qc_h5ad_file <- "data/object/qc.h5ad"

# Plotting Theme
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Import Seurat object

In [ ]:
so <- readRDS(raw_rds_file)

# Set meta data 

In [ ]:
so$time_point <- factor(so$time_point, levels=c("Baseline", "Tx", "D14", "D100", "GVHD"))
so$patient_id <- factor(so$patient_id, levels=paste0("Patient_", 1:15))
so$hto_demux_class <- factor(so$hto_demux_class, levels=c("Blood", "Skin", "Doublet", "Negative"))

# SCVI embedding and leiden cluster

In [ ]:
# Load python modules 
ad <- import("anndata")
pd <- import("pandas")
np <- import("numpy")

# Import adata
adata <- ad$read_h5ad("data/object/scvi/qc/adata.h5ad")

# Leiden cluster 
leiden <- py_to_r(adata$obs)[, "leiden", drop=FALSE]

# UMAP
umap <- py_to_r(adata$obsm['X_umap'])
rownames(umap) <-  py_to_r(adata$obs_names$to_list())
colnames(umap) <- c("UMAP_1", "UMAP_2")

In [ ]:
so <- AddMetaData(so, leiden)
so[["umap"]] <- CreateDimReducObject(embeddings=as.matrix(umap), key="UMAP_")

# Cellcycle

In [ ]:
cc <- read.csv("result/module/cellcycle/cellcycle.csv", row.names=1)
so <- AddMetaData(so, cc)

In [ ]:
options(repr.plot.width=15, repr.plot.height=5)

ggplot(so@meta.data, aes(x=patient_id, y=msS_RNA, color=msCC_class_RNA)) + geom_jitter() + theme(axis.text.x = element_text(angle=90, vjust=0.5, hjust=1)) + scale_color_manual(values=color$cc_phase_class)
ggplot(so@meta.data, aes(x=patient_id, y=msG2M_RNA, color=msCC_class_RNA)) + geom_jitter() + theme(axis.text.x = element_text(angle=90, vjust=0.5, hjust=1)) + scale_color_manual(values=color$cc_phase_class)

# CellTypist

In [ ]:
celltypist_low <- read.csv("result/celltypist/immune_all_low.csv", row.names=1)
colnames(celltypist_low) <- "celltypist_predicted_labels_low"

In [ ]:
celltypist_high <- read.csv("result/celltypist/immune_all_high.csv", row.names=1)
colnames(celltypist_high) <- "celltypist_predicted_labels_high"

In [ ]:
so <- AddMetaData(so, celltypist_low)
so <- AddMetaData(so, celltypist_high)

# Solo doublets 

In [ ]:
solo <- read.csv("result/solo/solo.csv", row.names=1)
so <- AddMetaData(so, solo)

In [ ]:
options(repr.plot.width=20, repr.plot.height=5)

ggplot(so@meta.data, aes(x=time_point, fill=solo_label)) + 
    geom_bar() + 
    xlab("sample [counts]") + ylab("") + ggtitle("HTO cell demultiplexing") + 
    facet_grid(~patient_id) + 
    scale_fill_manual(values=c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425")) + 
    scale_y_log10() + 
    theme(
        axis.text.x=element_text(angle=45, vjust=1, hjust=1), 
        legend.position="bottom"
    )

# HTO demultiplexing 

In [ ]:
options(repr.plot.width=20, repr.plot.height=5)

ggplot(so@meta.data, aes(x=time_point, fill=hto_demux_class)) + 
    geom_bar() + 
    xlab("sample [counts]") + ylab("") + ggtitle("HTO cell demultiplexing") + 
    facet_grid(~patient_id) + 
    scale_fill_manual(values=c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425")) + 
    scale_y_log10() + 
    theme(
        axis.text.x=element_text(angle=45, vjust=1, hjust=1), 
        legend.position="bottom"
    )

# Set Seurat object QC class

In [ ]:
# Set QC thresholds
so$nCount_RNA_min <- 1000
so$nCount_RNA_max <- max(so$nCount_RNA)

so$nFeature_RNA_min <- 700
so$nFeature_RNA_max <- max(so$nFeature_RNA)
    
so$pMT_RNA_min <- 0
so$pMT_RNA_max <- 7.5

so$pRP_RNA_min <- 5
so$pRP_RNA_max <- max(so$pRP_RNA)

so$pHB_RNA_min <- 0
so$pHB_RNA_max <- 1
    
so$qc_class <- ifelse(
        
    so$nCount_RNA >= so$nCount_RNA_min &
    so$nCount_RNA <= so$nCount_RNA_max & 
    
    so$nFeature_RNA >= so$nFeature_RNA_min &
    so$nFeature_RNA <= so$nFeature_RNA_max &
        
    so$pMT_RNA >= so$pMT_RNA_min & 
    so$pMT_RNA <= so$pMT_RNA_max &
    
    so$pRP_RNA >= so$pRP_RNA_min &
    so$pRP_RNA <= so$pRP_RNA_max &
    
    so$pHB_RNA >= so$pHB_RNA_min &
    so$pHB_RNA <= so$pHB_RNA_max &
    
    so$solo_label == "singlet" &
    
    so$hto_demux_class %in% c("Blood", "Skin"), 
    
    "pass", "fail"
        
)

# UMI and Feature count 

## Density plot

In [ ]:
density_plot_qc_1 <- density_plot_qc(so=so, title="Density plot UMI count", x=nCount_RNA, xlab="log10(UMI count)", min=nCount_RNA_min, max=nCount_RNA_max)
density_plot_qc_2 <- density_plot_qc(so=so, title="Density plot Feature count", x=nFeature_RNA, xlab="log10(Feature count)", min=nFeature_RNA_min, max=nFeature_RNA_max)
density_plot_qc_3 <- density_plot_qc(so=so, title="Density plot Mt %", x=pMT_RNA, xlab="Mt [%]", min=0, max=pMT_RNA_max, xlim=c(0,25), log10=FALSE)

In [ ]:
options(repr.plot.width=20, repr.plot.height=3*10)
density_plot_qc_1 + density_plot_qc_2 + density_plot_qc_3 + plot_layout(ncol=1) & theme(legend.position="none")

## Scattern plot

In [ ]:
scattern_plot_qc_1 <- scattern_plot_qc(so=so, title="Mitochondrial gene percentage", fill=pMT_RNA)
scattern_plot_qc_2 <- scattern_plot_qc(so=so, title="Hemoglobin gene percentage", fill=pHB_RNA)
scattern_plot_qc_3 <- scattern_plot_qc(so=so, title="Ribsonmal gene percentage", fill=pRP_RNA)
scattern_plot_qc_4 <- scattern_plot_qc(so=so, title="QC class", fill=qc_class) + scale_color_manual(values=c("fail"="#132B43", "pass"="#56B1F7"))
scattern_plot_qc_5 <- scattern_plot_qc(so=so, title="HTO demux class", fill=hto_demux_class) + scale_color_manual(values=c("Doublet"="#132B43", "Negative"="#132B43", "Blood"="#56B1F7", "Skin"="#56B1F7"))
scattern_plot_qc_6 <- scattern_plot_qc(so=so, title="SOLO doublet class", fill=solo_label) + scale_color_manual(values=c("doublet"="#132B43", "singlet"="#56B1F7"))

In [ ]:
options(repr.plot.width=20, repr.plot.height=6*10)
scattern_plot_qc_1 + scattern_plot_qc_2 + scattern_plot_qc_3 + scattern_plot_qc_4 + scattern_plot_qc_5 + scattern_plot_qc_6 + plot_layout(ncol=1) & theme(legend.position="bottom")

## Box plots 

In [ ]:
box_plot_qc_1 <- box_plot_qc(so=so, y=nCount_RNA, fill=time_point, ylab="UMI [count]", ymin=0)
box_plot_qc_2 <- box_plot_qc(so=so, y=nFeature_RNA, fill=time_point, ylab="Feature [count]", ymin=0)
box_plot_qc_3 <- box_plot_qc(so=so, y=pMT_RNA, fill=time_point, ylab="Mt [%]", ymin=0, ymax=100)
box_plot_qc_4 <- box_plot_qc(so=so, y=pHB_RNA, fill=time_point, ylab="Hb [%]", ymin=0, ymax=100)
box_plot_qc_5 <- box_plot_qc(so=so, y=pRP_RNA, fill=time_point, ylab="Rb [%]", ymin=0, ymax=100)

In [ ]:
options(repr.plot.width=20, repr.plot.height=5*7.5)
box_plot_qc_1[[1]] + box_plot_qc_1[[2]] + box_plot_qc_2[[1]]  + box_plot_qc_2[[2]] + 
box_plot_qc_3[[1]] + box_plot_qc_3[[2]] + box_plot_qc_4[[1]]  + box_plot_qc_4[[2]] + 
box_plot_qc_5[[1]] + box_plot_qc_5[[2]] + plot_layout(ncol=1)

# UMAP

In [ ]:
options(repr.plot.width=4*10, repr.plot.height=30)

dplot(so, group_by="leiden", shuffle=TRUE, label=TRUE, pt_size=2, na_color = "black") + 
dplot(so, group_by="hto_demux_class", shuffle=TRUE, label=TRUE, pt_size=2) + 
dplot(so, group_by="patient_id", shuffle=TRUE, label=TRUE, pt_size=2) + 
dplot(so, group_by="digestion_protocol", shuffle=TRUE, label=TRUE, pt_size=2) + 
dplot(so, group_by="qc_class", shuffle=TRUE, label=TRUE, pt_size=2) + 
dplot(so, group_by="hto_demux_class", shuffle=TRUE, label=TRUE, pt_size=2) + 
dplot(so, group_by="solo_label", shuffle=TRUE, label=TRUE, pt_size=2) + 

fplot(so, features="nCount_RNA", pt_size=2) + 
fplot(so, features="nFeature_RNA", pt_size=2) + 
fplot(so, features="pMT_RNA", pt_size=2) + 
fplot(so, features="pRP_RNA", pt_size=2) + 
fplot(so, features="pHB_RNA", pt_size=2) + 

patchwork::plot_layout(ncol=4)

# Filter cells by QC class

In [48]:
so <- subset(so, subset=qc_class=="pass")

# Filter genes 

In [49]:
cnt <- GetAssayData(so, assay="RNA", slot="counts")
cnt <- cnt[rowSums(cnt>0)>=1, ]

In [50]:
so <- CreateSeuratObject(counts=cnt, meta.data=so@meta.data, project="FD20250213HSCT")

# Save results

In [ ]:
saveRDS(so, file=qc_rds_file)

In [ ]:
# Store data as h5ad 
ad <- import("anndata")
pd <- import("pandas")
np <- import("numpy")
sp <- import("scipy")
    
# Transform dgCMatrix to sparse sc_sparse matrix
X <- GetAssayData(so, assay="RNA", layer="counts") %>% t()

# Combine 
adata <- ad$AnnData(X=sp$sparse$csr_matrix(X), obs=so@meta.data)
adata$var_names <- rownames(so)

adata$raw <- adata

# Save
adata$write_h5ad(qc_h5ad_file)

# Session info

In [ ]:
sessionInfo()